In [1]:
from pyspark.sql import functions as F

# 1. Читаємо очищені таблиці з lh_silver
df_stations = spark.table("lh_silver.dbo.stations")
df_sensors = spark.table("lh_silver.dbo.sensors")
df_readings = spark.table("lh_silver.dbo.readings")

# 2. Будуємо довідник станцій: dbo.dim_station
df_dim_station = (
    df_stations
    .select(
        F.col("station_id"),
        F.col("name").alias("station_name"),
        F.col("city"),
        F.col("country")
    )
    .dropDuplicates(["station_id"])
)

df_dim_station.write.format("delta").mode("overwrite").saveAsTable("dbo.dim_station")

# 3. Будуємо довідник сенсорів: dbo.dim_sensor
df_dim_sensor = (
    df_sensors
    .select("sensor_id", "metric", "unit")
    .dropDuplicates(["sensor_id"])
)

df_dim_sensor.write.format("delta").mode("overwrite").saveAsTable("dbo.dim_sensor")

# 4. Будуємо таблицю фактів: dbo.fact_readings (додаємо station_id через join зі сенсорами)
df_fact_readings = (
    df_readings.alias("r")
    .join(df_sensors.select("sensor_id", "station_id").alias("s"), on="sensor_id", how="inner")
    .select(
        F.col("r.reading_id"),
        F.col("r.sensor_id"),
        F.col("s.station_id"),
        F.col("r.measured_at"),
        F.col("r.measured_date"),
        F.col("r.value")
    )
)

df_fact_readings.write.format("delta").mode("overwrite").saveAsTable("dbo.fact_readings")

# 5. Перевіряємо створені таблиці
display(spark.table("dbo.dim_station"))
display(spark.table("dbo.dim_sensor"))
display(spark.table("dbo.fact_readings").limit(10))



StatementMeta(, 31bdc80d-8c6d-45cf-98e4-2811b9b7b12f, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bbf91f5f-1c3d-4106-bcf9-4d2c44f06197)

SynapseWidget(Synapse.DataFrame, c05e71ec-fe36-405e-96b2-fb4c7c0d0c74)

SynapseWidget(Synapse.DataFrame, 9c06176f-bd27-4ed2-8bf1-d212b738f339)